# 05 · Advanced RAG II: Retrieval
**Concepts:** (A) hybrid search: BM25 keywords + vectors, fused with Reciprocal Rank Fusion; (B) parent-document retrieval.
**TAC scenario:** search by symptom *and* by exact error code, then open the full article.

In [ ]:
from rag_common import MODEL, EMBED_MODEL, get_chat_model, get_vectorstore

model = get_chat_model()
vectorstore = get_vectorstore()          # opens the Chroma store built in Notebook 01
print(f"Chat: {MODEL} | Embeddings: {EMBED_MODEL} | chunks in store: {vectorstore._collection.count()}")

In [ ]:
import re
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from rag_common import load_chunks, golden_set, doc_ids, hit, reciprocal_rank, show

chunks = load_chunks()
STOP = {"what", "does", "do", "mean", "and", "should", "we", "about", "it", "the", "a", "an", "is", "how", "of", "to", "on", "in", "for", "our", "why", "which"}

def tokenize(text: str) -> list[str]:
    # keep codes like %ike-3-rekey_collision intact, strip trailing dots, drop filler words
    toks = (t.rstrip(".") for t in re.findall(r"[%\w][\w\-\.%]*", text.lower()))
    return [t for t in toks if t and t not in STOP]

print(tokenize("What does %STACK-2-SPLIT_DETECTED mean?"))
bm25 = BM25Retriever.from_documents(chunks, preprocess_func=tokenize, k=6)
vector = vectorstore.as_retriever(search_kwargs={"k": 6})
hybrid = EnsembleRetriever(retrievers=[bm25, vector], weights=[0.5, 0.5])   # Reciprocal Rank Fusion

## A. Hybrid search on error-code questions

In [ ]:
q = "What does %STACK-2-SPLIT_DETECTED mean?"
for name, r in [("BM25", bm25), ("vector", vector), ("hybrid", hybrid)]:
    print(f"{name:7s}", doc_ids(r.invoke(q))[:5])

In [ ]:
K = 4
by_type = {}
for g in golden_set():
    if g["type"] in ("error_code", "single_hop", "version_specific"):
        by_type.setdefault(g["type"], []).append(g)

print(f"hit@{K}           " + "  ".join(f"{t:>16s}" for t in by_type))
for name, r in [("BM25", bm25), ("vector", vector), ("hybrid", hybrid)]:
    row = []
    for t, gs in by_type.items():
        h = sum(hit(doc_ids(r.invoke(g["question"]))[:K], g["expected_doc_ids"]) for g in gs)
        row.append(f"{h:>2d}/{len(gs):<2d}".rjust(16))
    print(f"{name:16s}" + "  ".join(row))

> Keywords win on exact strings; vectors win on paraphrased symptoms; hybrid is robust on both. Tune `weights` per workload.

## B. Parent-document retrieval: match small, return big

In [ ]:
from langchain_chroma import Chroma
from langchain_core.stores import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.retrievers import ParentDocumentRetriever
from rag_common import load_markdown_dir, load_guides, get_embeddings

parents = load_markdown_dir("kb") + load_guides()           # whole KB articles and whole guide pages
child_store = Chroma(collection_name="pdr_children", embedding_function=get_embeddings())   # in-memory
pdr = ParentDocumentRetriever(
    vectorstore=child_store,
    docstore=InMemoryStore(),
    child_splitter=RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=0),
)
pdr.add_documents(parents)
print("parents:", len(parents), "| child chunks indexed:", child_store._collection.count())

In [ ]:
q = "Which command shows optical receive power on the HX-5000?"
child = child_store.similarity_search(q, k=1)[0]
parent = pdr.invoke(q)[0]
print(f"CHILD  ({len(child.page_content)} chars):\n{child.page_content}\n")
print(f"PARENT ({len(parent.page_content)} chars) [{parent.metadata['doc_id']}]:\n{parent.page_content[:700]}…")

---
**🔧 TAC takeaway:** Search by symptom AND by exact error code, then open the whole article, not just the search snippet. Hybrid and parent-document retrieval are those two habits.